# ResUNet DICOM denoiser — noise-conditioned version

Denoiser for mammography/tomosynthesis. The network is *conditioned on the noise
level* (FFDNet style): its input is the image **+ a channel that says how much to
denoise**. The denoising strength is therefore not fixed, and at inference it is
set with `CFG.INFER_LEVEL` without retraining. This avoids the fidelity-vs-texture
tug of war in the loss that pushed towards oversmoothing.

Residual convention: the network predicts the **noise**, `clean = noisy - noise`.
The final conv is zero-initialized, so at step 0 it predicts zero noise (baseline = pass-through).

Run the cells top to bottom. Resilient to disconnections (auto-resume).

## Setup (install + import + mount Drive)

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pydicom", "torchmetrics", "scikit-image"], check=True)
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        if "already contain files" in str(e):
            drive.mount("/content/drive", force_remount=True)
        else:
            raise

import os, glob, json, time, random, copy, math, shutil
import numpy as np
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from skimage import filters, morphology, measure

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
status = f"OK setup | device={device}"
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    status += f" | gpu={p.name} | vram={p.total_memory / 1e9:.1f}GB"
print(status)

## Configuration

In [ ]:
class CFG:
    DRIVE_WORK_DIR  = "/content/drive/MyDrive/resunet"
    DRIVE_DATA_DIR  = "/content/drive/MyDrive/dataset"
    CKPT_DIR        = f"{DRIVE_WORK_DIR}/checkpoints"
    LOG_DIR         = f"{DRIVE_WORK_DIR}/logs"
    OUT_DIR         = f"{DRIVE_WORK_DIR}/outputs_dicom"
    PREVIEW_DIR     = f"{DRIVE_WORK_DIR}/previews"
    SPLIT_FILE      = f"{DRIVE_WORK_DIR}/splits.json"
    MASK_CACHE_DIR  = f"{DRIVE_WORK_DIR}/masks"
    LOCAL_DATA_DIR  = "/content/dataset"

    PATCH_SIZE          = 256
    VAL_SPLIT           = 0.10
    TEST_SPLIT          = 0.05
    NUM_WORKERS         = 2
    MIN_BREAST_FRACTION = 0.30
    MAX_REJECT_ATTEMPTS = 25
    N_CROPS_TRAIN       = 16
    N_CROPS_VAL         = 2

    # --- tomo-NPS noise with VARIABLE AMPLITUDE (needed for conditioning) ---
    # training spans a wide range of 'level': the network learns to denoise at
    # different strengths, not a single one. At inference you pick the strength (INFER_LEVEL).
    TOMO_LEVEL_MIN  = 0.002
    TOMO_LEVEL_MAX  = 0.020
    TOMO_READ_MIN   = 0.0005
    TOMO_READ_MAX   = 0.0015
    IDENTITY_PROB   = 0.05      # sometimes zero noise (level=0): anchors the "do not denoise" case

    # --- noise conditioning (FFDNet style) ---
    SIGMA_GAIN      = 50.0      # sigma channel gain: level~0.01 -> ~0.5 (salient for the network)
    INFER_LEVEL     = 0.003     # denoising strength at inference, value used for the reference results

    BASE_CHANNELS   = 48
    BATCH_SIZE      = 8
    EPOCHS          = 120
    LR              = 2e-4
    WEIGHT_DECAY    = 1e-4
    GRAD_CLIP       = 1.0
    USE_AMP         = True
    WARMUP_EPOCHS   = 3
    EARLY_STOP_PAT  = 25

    KEEP_SNAPSHOTS      = 3
    RESUME              = True
    PREVIEW_EVERY_EP    = 5

for d in [CFG.DRIVE_WORK_DIR, CFG.CKPT_DIR, CFG.LOG_DIR,
          CFG.OUT_DIR, CFG.PREVIEW_DIR, CFG.MASK_CACHE_DIR,
          CFG.LOCAL_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(
    f"OK config | epochs={CFG.EPOCHS} | batch={CFG.BATCH_SIZE} | patch={CFG.PATCH_SIZE} | "
    f"level=[{CFG.TOMO_LEVEL_MIN},{CFG.TOMO_LEVEL_MAX}] | infer_level={CFG.INFER_LEVEL} | "
    f"base_ch={CFG.BASE_CHANNELS} | lr={CFG.LR} | resume={CFG.RESUME}"
)

## Dataset copy (Drive → local, resilient)

In [ ]:
def _remount_drive():
    if not IN_COLAB:
        return
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(2)
    try:
        drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        print(f"    remount failed: {e}")

def _copy_atomic(src, dst, max_retries=4):
    tmp = dst + ".tmp"
    for attempt in range(max_retries):
        try:
            if os.path.exists(tmp):
                try: os.remove(tmp)
                except OSError: pass
            shutil.copy2(src, tmp)
            os.replace(tmp, dst)
            return True
        except OSError as e:
            last_err = e
            if attempt < max_retries - 1:
                print(f"retry {attempt + 1}/{max_retries} | remount Drive")
                _remount_drive()
                time.sleep(3 * (attempt + 1))
    try: os.remove(tmp)
    except OSError: pass
    return False

drive_files = sorted(glob.glob(
    os.path.join(CFG.DRIVE_DATA_DIR, "**", "*.dcm"), recursive=True))
print(f"Dataset Drive | dicom={len(drive_files)}")
assert len(drive_files) > 0, (
    f"Drive is empty at {CFG.DRIVE_DATA_DIR} — check the path and the mount."
)

MIN_VALID_BYTES = 1_000_000

def _is_already_copied(dst):
    return os.path.exists(dst) and os.path.getsize(dst) > MIN_VALID_BYTES

MAX_GLOBAL_PASSES = 4
remaining = list(drive_files)
n_done    = 0

for pass_idx in range(MAX_GLOBAL_PASSES):
    if not remaining:
        break
    print(f"Dataset copy | pass={pass_idx + 1}/{MAX_GLOBAL_PASSES} | missing={len(remaining)}")

    still = []
    for i, src in enumerate(remaining):
        dst = os.path.join(CFG.LOCAL_DATA_DIR, os.path.basename(src))
        if _is_already_copied(dst):
            n_done += 1
            continue
        if _copy_atomic(src, dst):
            n_done += 1
            if n_done % 50 == 0:
                print(f"copied={n_done}/{len(drive_files)}")
        else:
            still.append(src)

    if len(still) == len(remaining):
        print("no progress | remount in 15s")
        time.sleep(15)
        _remount_drive()
    remaining = still

local_files = sorted(glob.glob(os.path.join(CFG.LOCAL_DATA_DIR, "*.dcm")))
print(f"OK local dataset | dicom={len(local_files)}/{len(drive_files)}")
if remaining:
    print(f"WARNING incomplete dataset | missing={len(remaining)}")

## DICOM helpers (normalized load/save, fixed pmax)

In [ ]:
def get_pixel_max(ds):
    return float((1 << int(getattr(ds, "BitsStored", 16))) - 1)

def _dcmread_robust(path, max_retries=4):
    for attempt in range(max_retries):
        try:
            return pydicom.dcmread(path)
        except OSError as e:
            if attempt == max_retries - 1:
                raise
            wait = 2.0 * (2 ** attempt)
            print(f"  OSError on {os.path.basename(path)}: {e} "
                  f"— retrying in {wait:.0f}s ({attempt+1}/{max_retries})")
            time.sleep(wait)

def load_dicom_normalized(path):
    ds   = _dcmread_robust(path)
    arr  = ds.pixel_array.astype(np.float32)
    pmax = get_pixel_max(ds)
    mono1 = (getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1")
    if mono1:
        arr = pmax - arr
    arr = np.clip(arr / pmax, 0.0, 1.0)
    return arr, {"pmax": pmax, "mono1": mono1, "shape": arr.shape, "path": path}

def save_dicom_from_norm(arr_norm, source_path, out_path):
    """Save arr_norm [0,1] as DICOM, cloning the metadata of the source.
    Rescales with the FIXED pmax (bit depth), NOT per-image min/max."""
    ds_src = _dcmread_robust(source_path)
    ds_out = copy.deepcopy(ds_src)
    pmax   = get_pixel_max(ds_src)
    mono1  = (getattr(ds_src, "PhotometricInterpretation", "") == "MONOCHROME1")
    
    arr_int = np.clip(np.round(arr_norm * pmax), 0, pmax).astype(np.uint16)
    if mono1:
        arr_int = (pmax - arr_int.astype(np.float32)).astype(np.uint16)

    ds_out.PixelData            = arr_int.tobytes()
    ds_out.Rows, ds_out.Columns = arr_int.shape
    ds_out.BitsAllocated        = 16
    ds_out.BitsStored           = int(getattr(ds_src, "BitsStored", 16))
    ds_out.HighBit              = ds_out.BitsStored - 1
    ds_out.PixelRepresentation  = 0
    ds_out.SamplesPerPixel      = 1
    ds_out.file_meta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian
    try:
        ds_out.save_as(out_path, enforce_file_format=True)
    except TypeError:
        ds_out.save_as(out_path, write_like_original=False)

## Breast mask (Otsu + connected component, cached on disk)

In [ ]:
from pathlib import Path

def compute_breast_mask(arr_norm, min_intensity=0.02):
    if arr_norm.max() < min_intensity:
        return np.zeros_like(arr_norm, dtype=bool)
    try:
        thr = filters.threshold_otsu(arr_norm)
    except Exception:
        thr = float(arr_norm.mean()) # fall back to the mean if Otsu fails
    mask = arr_norm > max(thr, min_intensity)
    labels = measure.label(mask)  # one label per connected group, to tell objects apart
    if labels.max() == 0:   
        return mask.astype(bool)
    counts    = np.bincount(labels.ravel())
    counts[0] = 0
    mask = (labels == np.argmax(counts))
    mask = morphology.remove_small_holes(mask, area_threshold=5000)
    mask = morphology.binary_closing(mask, morphology.disk(7))
    return mask.astype(bool)

def _mask_cache_path(dicom_path):
    stem = Path(dicom_path).stem  # file name without extension
    return os.path.join(CFG.MASK_CACHE_DIR, stem + ".npz")

def load_or_compute_mask(dicom_path, arr_norm=None):
    cp = _mask_cache_path(dicom_path)
    if os.path.exists(cp):
        d = np.load(cp)
        H, W = int(d["H"]), int(d["W"])
        return np.unpackbits(d["packed"])[: H * W].reshape(H, W).astype(bool)
    if arr_norm is None:
        arr_norm, _ = load_dicom_normalized(dicom_path)
    mask = compute_breast_mask(arr_norm)
    H, W = mask.shape
    np.savez_compressed(cp, packed=np.packbits(mask), H=H, W=W)
    return mask

## Train/val/test split (fixed, saved on Drive)

In [ ]:
dicom_paths = sorted(glob.glob(os.path.join(CFG.LOCAL_DATA_DIR, "*.dcm")))
assert len(dicom_paths) > 0, f"No DICOM in {CFG.LOCAL_DATA_DIR}: run the dataset copy first."
print(f"OK dataset | dicom={len(dicom_paths)}")

def build_or_load_split(all_paths, val_frac, test_frac, split_file, seed=SEED):
    all_set = set(all_paths)
    if os.path.exists(split_file):
        with open(split_file) as f:
            sp = json.load(f)
        stored = set(sp["train"]) | set(sp["val"]) | set(sp["test"])
        if stored == all_set:
            print(f"Split | loaded | train={len(sp['train'])} | val={len(sp['val'])} | test={len(sp['test'])}")
            return sp["train"], sp["val"], sp["test"]
    shuffled = list(all_paths)
    random.Random(seed).shuffle(shuffled)
    n      = len(shuffled)
    n_test = int(round(n * test_frac))
    n_val  = int(round(n * val_frac))
    test   = sorted(shuffled[:n_test])
    val    = sorted(shuffled[n_test : n_test + n_val])
    train  = sorted(shuffled[n_test + n_val :])
    with open(split_file, "w") as f:
        json.dump({"train": train, "val": val, "test": test, "seed": seed}, f, indent=1)
    print(f"Split | saved | train={len(train)} | val={len(val)} | test={len(test)}")
    return train, val, test

train_paths, val_paths, test_paths = build_or_load_split(
    dicom_paths, CFG.VAL_SPLIT, CFG.TEST_SPLIT, CFG.SPLIT_FILE)

## Synthetic tomo-NPS noise + conditioning

The noise has the real tomo grain (radial NPS measured on a real projection) and a
signal-dependent amplitude. `__call__` also returns the **`level`** used, which
becomes the conditioning channel. `make_input` packs image + sigma channel.

In [ ]:
NPS_RADIAL = [0.000, 0.012, 0.082, 0.270, 0.322, 0.588, 0.944, 0.680, 1.104, 1.175,
              1.331, 1.172, 1.234, 1.151, 1.250, 1.138, 1.294, 1.251, 1.266, 1.247,
              1.284, 1.235, 1.074, 1.227, 1.223, 1.109, 1.145, 1.066, 1.033, 1.103,
              1.162, 1.139, 1.125, 1.050, 0.995, 0.964, 1.016, 0.941, 0.935, 1.014,
              0.993, 1.015, 0.949, 0.887, 0.866, 1.002, 0.945, 0.993]

#Noise Power Spectrum

class TomoNPSNoise:
    """Tomo-like noise: grain = real NPS + signal-dependent amplitude.
    sigma(clean) = sqrt(read^2 + level^2 * (1 - clean)).
    Returns (noisy, level) where level is (B,1,1,1), used for conditioning."""
    def __init__(self, level_min, level_max, read_min, read_max, nps_radial=NPS_RADIAL):
        self.level_min = level_min; self.level_max = level_max # signal-dependent part, larger on dark pixels
        self.read_min  = read_min;  self.read_max  = read_max  # image-independent, simulates the detector
        self.nps = torch.tensor(nps_radial, dtype=torch.float32)
        self._cache = {}

    def _freq_response(self, H, W, device):
        key = (H, W, device)
        if key not in self._cache:

            # all frequencies
            fy = torch.fft.fftfreq(H, device=device)
            fx = torch.fft.fftfreq(W, device=device)

            # map from Cartesian frequencies to radial distance
            FR = torch.sqrt(fy[:, None] ** 2 + fx[None, :] ** 2) * 2.0   # Nyquist -> 1

            nps = self.nps.to(device)

            # index into the NPS: i0 is the integer part, frac the fractional part
            pos = FR.clamp(0, 1) * (len(nps) - 1)
            i0  = pos.long().clamp(0, len(nps) - 2)
            frac = pos - i0
            
            # linear interpolation of the curve, to avoid steps
            val = nps[i0] * (1 - frac) + nps[i0 + 1] * frac

            # the filter acts on amplitude, power = amplitude^2, hence the sqrt
            self._cache[key] = torch.sqrt(val.clamp(min=0))

        return self._cache[key]

    def _colored(self, shape, device):

        # noise with the NPS structure
        A = self._freq_response(shape[-2], shape[-1], device)[None, None]

        # white noise
        w = torch.randn(shape, device=device)

        # apply the filter
        out = torch.fft.ifft2(torch.fft.fft2(w) * A).real

        return out / out.std(dim=(-2, -1), keepdim=True).clamp(min=1e-8)

    def __call__(self, clean):
        c = clean.clamp(0, 1)
        B = c.shape[0]
        dev = c.device

        # global noise level, one random value per image
        level = torch.empty(B, 1, 1, 1, device=dev).uniform_(self.level_min, self.level_max)
        # read noise, also one per image
        read = torch.empty(B, 1, 1, 1, device=dev).uniform_(self.read_min, self.read_max)

        # per-pixel noise std: read^2 is the fixed part, level^2*(1-c) the
        # signal-dependent part (darker pixels -> more noise)
        sigma = torch.sqrt(read ** 2 + level ** 2 * (1.0 - c).clamp(min=0))

        n = self._colored(c.shape, dev)
        noisy = (c + n * sigma).clamp(0, 1)

        # level is returned because it becomes the conditioning channel
        return noisy, level


noise_adder = TomoNPSNoise(
    CFG.TOMO_LEVEL_MIN, CFG.TOMO_LEVEL_MAX, CFG.TOMO_READ_MIN, CFG.TOMO_READ_MAX)


def make_input(noisy, level):
    """Build the 2-channel input: [noisy_image, sigma_channel].
    level: tensor (B,1,1,1) or scalar (float). The sigma channel is
    constant = level * SIGMA_GAIN (numerical salience)."""
    if not torch.is_tensor(level):
        level = torch.full((noisy.shape[0], 1, 1, 1), float(level),
                           device=noisy.device, dtype=noisy.dtype)
    sig = (level * CFG.SIGMA_GAIN).expand(noisy.shape[0], 1, noisy.shape[-2], noisy.shape[-1])
    return torch.cat([noisy, sig], dim=1)

print(f"OK noise | conditioned tomo_nps | grain=real NPS ({len(NPS_RADIAL)} pt) | "
      f"sigma_gain={CFG.SIGMA_GAIN}")

## Dataset and loader

In [ ]:
class MammoDenoiseDataset(Dataset):
    def __init__(self, paths, patch, n_crops, train):
        self.paths   = paths
        self.patch   = patch
        self.n_crops = n_crops
        self.train   = train

    def __len__(self):
        return len(self.paths) * self.n_crops

    def _load(self, idx):
        path = self.paths[idx % len(self.paths)] # the modulo maps each crop index back to its image
        arr, _ = load_dicom_normalized(path)
        mask   = load_or_compute_mask(path, arr)
        return arr, mask

    def _sample_crop(self, arr, mask):
        H, W = arr.shape
        p    = self.patch

        # image resized bilinearly, mask with nearest neighbour
        if H < p or W < p:
            t = F.interpolate(torch.from_numpy(arr)[None, None],
                              size=(p, p), mode="bilinear", align_corners=False)
            m = F.interpolate(torch.from_numpy(mask.astype(np.float32))[None, None],
                              size=(p, p), mode="nearest")
            return t.squeeze().numpy(), (m.squeeze().numpy() > 0.5)

        if self.train:
            for _ in range(CFG.MAX_REJECT_ATTEMPTS):

                # random point
                y = np.random.randint(0, H - p + 1)
                x = np.random.randint(0, W - p + 1)

                # fraction of breast in the crop
                if mask[y:y+p, x:x+p].mean() >= CFG.MIN_BREAST_FRACTION: 
                    # accept it, to avoid empty patches
                    return arr[y:y+p, x:x+p], mask[y:y+p, x:x+p]

            # no valid crop found: center it on a random breast pixel
            ys, xs = np.where(mask)
            i = np.random.randint(len(ys)) if len(ys) else 0
            cy, cx = (int(ys[i]), int(xs[i])) if len(ys) else (H // 2, W // 2)
        else:
            # outside training, use the breast centroid
            ys, xs = np.where(mask)
            cy, cx = ((int(ys.mean()), int(xs.mean())) if len(ys)
                      else (H // 2, W // 2))

        y = int(np.clip(cy - p // 2, 0, H - p))
        x = int(np.clip(cx - p // 2, 0, W - p))
        return arr[y:y+p, x:x+p], mask[y:y+p, x:x+p]

    # Random flips and rotations, for variety.
    def _augment(self, arr, mask):
        if np.random.rand() < 0.5:
            arr  = np.ascontiguousarray(arr[:, ::-1])
            mask = np.ascontiguousarray(mask[:, ::-1])
        if np.random.rand() < 0.5:
            arr  = np.ascontiguousarray(arr[::-1, :])
            mask = np.ascontiguousarray(mask[::-1, :])
        k = np.random.randint(4)
        if k:
            arr  = np.ascontiguousarray(np.rot90(arr, k))
            mask = np.ascontiguousarray(np.rot90(mask, k))
        return arr, mask

    def __getitem__(self, idx):
        arr, mask = self._load(idx)
        crop, cmask = self._sample_crop(arr, mask)
        if self.train:
            crop, cmask = self._augment(crop, cmask)
        return (torch.from_numpy(crop.copy()).unsqueeze(0).float(),
                torch.from_numpy(cmask.copy().astype(np.float32)).unsqueeze(0))


train_ds = MammoDenoiseDataset(train_paths, CFG.PATCH_SIZE, CFG.N_CROPS_TRAIN, True)
val_ds   = MammoDenoiseDataset(val_paths,   CFG.PATCH_SIZE, CFG.N_CROPS_VAL,   False)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True,
                          drop_last=True,
                          persistent_workers=(CFG.NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True,
                          persistent_workers=(CFG.NUM_WORKERS > 0))

print(f"OK dataloader | train_crop={len(train_ds)} | val_crop={len(val_ds)} | "
      f"train_batch={len(train_loader)} | val_batch={len(val_loader)}")

## Architecture — conditioned ResUNet (in_ch=2)

A residual ResUNet whose first block takes **2 channels** (image + sigma
channel). Zero-initialized head → pass-through baseline preserved.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        g = min(8, out_ch)
        while out_ch % g != 0:
            g //= 2
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(g, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(g, out_ch)
        self.skip  = (nn.Conv2d(in_ch, out_ch, 1, bias=False)
                      if in_ch != out_ch else nn.Identity())
        self.act   = nn.SiLU(inplace=True)

    def forward(self, x):
        h = self.act(self.norm1(self.conv1(x)))
        h = self.norm2(self.conv2(h))
        return self.act(h + self.skip(x))


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool  = nn.AvgPool2d(2)
        self.block = ResBlock(in_ch, out_ch)

    def forward(self, x):
        return self.block(self.pool(x))


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up    = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.block = ResBlock(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:],
                              mode="bilinear", align_corners=False)
        return self.block(torch.cat([x, skip], dim=1))


class ResUNet(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=48):
        super().__init__()
        b = base
        self.enc0 = ResBlock(in_ch, b)
        self.enc1 = Down(b,    b*2)
        self.enc2 = Down(b*2,  b*4)
        self.enc3 = Down(b*4,  b*8)
        self.enc4 = Down(b*8,  b*8)
        self.dec3 = Up(b*8,  b*8, b*4)
        self.dec2 = Up(b*4,  b*4, b*2)
        self.dec1 = Up(b*2,  b*2, b)
        self.dec0 = Up(b,    b,   b)
        self.head = nn.Conv2d(b, out_ch, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d  = self.dec3(e4, e3)
        d  = self.dec2(d,  e2)
        d  = self.dec1(d,  e1)
        d  = self.dec0(d,  e0)
        return self.head(d)


model = ResUNet(in_ch=2, out_ch=1, base=CFG.BASE_CHANNELS).to(device)
nparams = sum(p.numel() for p in model.parameters())
print(f"OK architecture | conditioned ResUNet | in_ch=2 | base={CFG.BASE_CHANNELS} | "
      f"params={nparams / 1e6:.2f}M")

## Loss and metrics

A **simple** loss: Charbonnier masked on the estimated noise. Conditioning handles
"how much to denoise", so the texture term that used to create a tug of war is not
needed. `hf` and `ssim` are logged as **diagnostics** only, NOT targets (the clean
image already has its own noise → hf<1 is normal).

In [ ]:
def masked_charbonnier(pred, target, mask, eps=1e-3):
    diff = torch.sqrt((pred - target) ** 2 + eps ** 2)
    return (diff * mask).sum() / mask.sum().clamp(min=1.0)

def masked_psnr(pred, target, mask, data_range=1.0):
    mse = (((pred - target) ** 2) * mask).sum() / mask.sum().clamp(min=1.0)
    return 10.0 * torch.log10(data_range ** 2 / mse.clamp(min=1e-12))

def _grad_xy(t):
    dy = t[..., 1:, :] - t[..., :-1, :]
    dx = t[..., :, 1:] - t[..., :, :-1]
    return dy, dx

def masked_hf_ratio(pred, target, mask):
    """DIAGNOSTIC (not a target): high-frequency energy denoised/clean in the breast.
    ~1 texture preserved, <1 oversmoothing, >1 residual noise."""
    dyp, dxp = _grad_xy(pred); dyt, dxt = _grad_xy(target)
    my = mask[..., 1:, :] * mask[..., :-1, :]
    mx = mask[..., :, 1:] * mask[..., :, :-1]
    ep = (dyp.pow(2) * my).sum() + (dxp.pow(2) * mx).sum()
    et = (dyt.pow(2) * my).sum() + (dxt.pow(2) * mx).sum()
    return (ep / et.clamp(min=1e-12)).sqrt()

ssim_fn = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

def masked_ssim(pred, target, mask):
    """SSIM on the per-sample breast bounding box (avoids the background inflating it)."""
    vals = []
    for b in range(pred.shape[0]):
        idx = mask[b, 0].nonzero()
        if idx.numel() == 0:
            vals.append(ssim_fn(pred[b:b+1].clamp(0, 1).float(), target[b:b+1].float()))
            continue
        y0, x0 = idx[:, 0].min().item(), idx[:, 1].min().item()
        y1, x1 = idx[:, 0].max().item() + 1, idx[:, 1].max().item() + 1
        p = pred[b:b+1, :, y0:y1, x0:x1].clamp(0, 1).float()
        t = target[b:b+1, :, y0:y1, x0:x1].float()
        vals.append(ssim_fn(p, t))
    return torch.stack(vals).mean()

def compute_loss(noise_pred, noise_true, mask):
    return masked_charbonnier(noise_pred, noise_true, mask)

print("OK loss | masked Charbonnier | diagnostics: psnr/ssim/hf")

## Optimizer + scheduler (warmup + cosine) + AMP

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),
                              lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

def _lr_lambda(epoch):
    if epoch < CFG.WARMUP_EPOCHS:
        return (epoch + 1) / max(1, CFG.WARMUP_EPOCHS)
    t = (epoch - CFG.WARMUP_EPOCHS) / max(1, CFG.EPOCHS - CFG.WARMUP_EPOCHS)
    return 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * min(t, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=_lr_lambda)
scaler    = torch.amp.GradScaler("cuda", enabled=CFG.USE_AMP)
print("OK optimizer | AdamW + warmup/cosine + AMP")

## Checkpoint + resume (atomic)

In [ ]:
BEST_CKPT = os.path.join(CFG.CKPT_DIR, "best.pt")
LAST_CKPT = os.path.join(CFG.CKPT_DIR, "last.pt")
LOG_CSV   = os.path.join(CFG.LOG_DIR,  "training_log.csv")

def save_ckpt(path, epoch, model, optimizer, scheduler, scaler, best_val, history):
    tmp = path + ".tmp"
    torch.save({"epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler":    scaler.state_dict(),
                "best_val":  best_val,
                "history":   history}, tmp)
    os.replace(tmp, path)

def load_ckpt(path, model, optimizer=None, scheduler=None, scaler=None):
    ck = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    if optimizer  is not None: optimizer.load_state_dict(ck["optimizer"])
    if scheduler  is not None: scheduler.load_state_dict(ck["scheduler"])
    if scaler     is not None: scaler.load_state_dict(ck["scaler"])
    return ck.get("epoch", -1), ck.get("best_val", 0.0), ck.get("history", [])

def prune_snapshots(keep_n):
    snaps = sorted(glob.glob(os.path.join(CFG.CKPT_DIR, "epoch_*.pt")))
    for p in snaps[:-keep_n]:
        try: os.remove(p)
        except OSError: pass

start_epoch = 0
best_val = 0.0   # selected on val_psnr: higher is better
history = []
resume_path = LAST_CKPT if os.path.exists(LAST_CKPT) else BEST_CKPT

if CFG.RESUME and os.path.exists(resume_path):
    try:
        ep, best_val, history = load_ckpt(resume_path, model, optimizer, scheduler, scaler)
        start_epoch = ep + 1
        print(f"OK checkpoint | loaded={os.path.basename(resume_path)} | "
              f"start_epoch={start_epoch} | best_psnr={best_val:.4f}")
    except Exception as e:
        print(f"WARNING checkpoint not loaded | {type(e).__name__}: {e}")
        start_epoch, best_val, history = 0, 0.0, []
else:
    print(f"Checkpoint | no resume | expected={BEST_CKPT}")

if not os.path.exists(LOG_CSV):
    with open(LOG_CSV, "w") as f:
        f.write("epoch,train_loss,train_psnr,val_loss,val_psnr,val_ssim,val_hf,lr,time_s\n")

## Training

For each batch: noise on the fly with a random `level` → 2-channel input via
`make_input` → the network predicts the noise. Best checkpoint selected on **val_psnr**.
`hf` is logged as a diagnostic (watch for oversmoothing, but it is not the criterion).

In [ ]:
def _run_epoch(loader, train_mode, epoch_idx):
    model.train(train_mode)
    tot_loss = tot_psnr = tot_ssim = tot_hf = 0.0
    n = len(loader)
    for i, (clean, mask) in enumerate(loader):
        clean = clean.to(device, non_blocking=True)
        mask  = mask.to(device,  non_blocking=True)
        if not train_mode:
            torch.manual_seed(20240000 + i)
            np.random.seed(20240000 + i)

        if train_mode and np.random.rand() < CFG.IDENTITY_PROB:
            noisy = clean
            level = torch.zeros(clean.shape[0], 1, 1, 1, device=device)
        else:
            noisy, level = noise_adder(clean)
        noise_true = noisy - clean

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=CFG.USE_AMP):
                noise_pred = model(make_input(noisy, level))
                clean_pred = noisy - noise_pred
                loss = compute_loss(noise_pred, noise_true, mask)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

            with torch.no_grad():
                cp  = clean_pred.clamp(0, 1).float()
                p_v = masked_psnr(cp, clean.float(), mask).item()
                if not train_mode:
                    s_v = masked_ssim(cp, clean.float(), mask).item()
                    h_v = masked_hf_ratio(cp, clean.float(), mask).item()
                else:
                    s_v = h_v = 0.0

        tot_loss += loss.item(); tot_psnr += p_v; tot_ssim += s_v; tot_hf += h_v

    return tot_loss / n, tot_psnr / n, tot_ssim / n, tot_hf / n


def _save_preview(epoch_idx):
    if not val_paths:
        return
    path  = val_paths[epoch_idx % len(val_paths)]
    arr,_ = load_dicom_normalized(path)
    m     = load_or_compute_mask(path, arr)
    H, W  = arr.shape; p = CFG.PATCH_SIZE
    ys, xs = np.where(m)
    cy = int(ys.mean()) if len(ys) else H // 2
    cx = int(xs.mean()) if len(xs) else W // 2
    y  = int(np.clip(cy - p // 2, 0, max(H - p, 0)))
    x  = int(np.clip(cx - p // 2, 0, max(W - p, 0)))

    ct = torch.from_numpy(arr[y:y+p, x:x+p]).unsqueeze(0).unsqueeze(0).float().to(device)
    torch.manual_seed(7777); np.random.seed(7777)
    noisy_t, level = noise_adder(ct)
    was_train = model.training; model.eval()
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=CFG.USE_AMP):
            den = (noisy_t - model(make_input(noisy_t, level))).clamp(0, 1).float()
    if was_train: model.train()

    full_n = arr.copy(); full_d = arr.copy()
    full_n[y:y+p, x:x+p] = noisy_t.squeeze().cpu().numpy()
    full_d[y:y+p, x:x+p] = den.squeeze().cpu().numpy()

    pdir = os.path.join(CFG.PREVIEW_DIR, f"epoch_{epoch_idx+1:03d}")
    os.makedirs(pdir, exist_ok=True)
    save_dicom_from_norm(full_n, path, os.path.join(pdir, "noisy.dcm"))
    save_dicom_from_norm(full_d, path, os.path.join(pdir, "denoised.dcm"))


patience = 0
print(f"Training | start_epoch={start_epoch} | remaining={CFG.EPOCHS - start_epoch} | best_psnr={best_val:.4f}", flush=True)

try:
    for epoch in range(start_epoch, CFG.EPOCHS):
        t0 = time.time()
        tr_loss, tr_psnr, _, _          = _run_epoch(train_loader, True,  epoch)
        va_loss, va_psnr, va_ssim, va_hf = _run_epoch(val_loader,   False, epoch)
        scheduler.step()
        lr_now = optimizer.param_groups[0]["lr"]
        el = time.time() - t0

        print(
            f"epoch={epoch+1:03d}/{CFG.EPOCHS} | "
            f"tr {tr_loss:.5f} {tr_psnr:.2f}dB | "
            f"va {va_loss:.5f} {va_psnr:.2f}dB ssim={va_ssim:.3f} hf={va_hf:.3f} | "
            f"lr={lr_now:.1e} | {el:.0f}s",
            flush=True
        )

        history.append({"epoch": epoch+1, "train_loss": tr_loss,
                        "train_psnr": tr_psnr, "val_loss": va_loss,
                        "val_psnr": va_psnr, "val_ssim": va_ssim,
                        "val_hf": va_hf, "lr": lr_now, "time_s": el})
        with open(LOG_CSV, "a") as f:
            f.write(f"{epoch+1},{tr_loss:.6f},{tr_psnr:.4f},{va_loss:.6f},"
                    f"{va_psnr:.4f},{va_ssim:.4f},{va_hf:.4f},{lr_now:.2e},{el:.1f}\n")

        save_ckpt(LAST_CKPT, epoch, model, optimizer,
                  scheduler, scaler, best_val, history)

        if va_psnr > best_val:
            best_val = va_psnr
            save_ckpt(BEST_CKPT, epoch, model, optimizer,
                      scheduler, scaler, best_val, history)
            patience = 0
            print(f"checkpoint=best | epoch={epoch+1} | val_psnr={best_val:.4f}", flush=True)
        else:
            patience += 1
            print(f"patience={patience}/{CFG.EARLY_STOP_PAT}", flush=True)

        if (epoch + 1) % CFG.PREVIEW_EVERY_EP == 0:
            try:
                _save_preview(epoch)
                print(f"preview=epoch_{epoch+1:03d}", flush=True)
            except Exception as e:
                print(f"preview=failed | {type(e).__name__}: {e}", flush=True)

        snap = os.path.join(CFG.CKPT_DIR, f"epoch_{epoch+1:03d}.pt")
        save_ckpt(snap, epoch, model, optimizer,
                  scheduler, scaler, best_val, history)
        prune_snapshots(CFG.KEEP_SNAPSHOTS)

        if patience >= CFG.EARLY_STOP_PAT:
            print(f"Early stopping at epoch {epoch+1}.", flush=True)
            break

    print("OK training done", flush=True)

except KeyboardInterrupt:
    print("Interrupted | checkpoint=last.pt", flush=True)
except Exception as e:
    import traceback; traceback.print_exc()
    print("Error | checkpoint=last.pt", flush=True)

## Sliding-window inference (conditioned)

Pass `level` = denoising strength. Full images are split into 256 patches with
overlap and Gaussian blending. The same `level` is used on all patches.

In [ ]:
def sliding_window_inference(model, image_t, level, patch=256, overlap=64):
    """image_t: (1,1,H,W) on device; level: denoising strength (float or tensor) → clean_pred."""
    model.eval()
    _, _, H, W = image_t.shape
    step  = patch - overlap
    pad_h = (step - max(H - patch, 0) % step) % step if H >= patch else patch - H
    pad_w = (step - max(W - patch, 0) % step) % step if W >= patch else patch - W
    xp    = F.pad(image_t, (0, pad_w, 0, pad_h), mode="reflect")
    Hp, Wp = xp.shape[-2:]

    out = torch.zeros_like(xp)
    wts = torch.zeros_like(xp)

    sigma  = patch / 6.0
    coords = torch.arange(patch, dtype=torch.float32, device=image_t.device) - patch / 2.0
    g      = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    win    = (g.unsqueeze(0) * g.unsqueeze(1)).unsqueeze(0).unsqueeze(0)

    ys = list(range(0, Hp - patch + 1, step))
    xs = list(range(0, Wp - patch + 1, step))
    if not ys or ys[-1] != Hp - patch: ys.append(Hp - patch)
    if not xs or xs[-1] != Wp - patch: xs.append(Wp - patch)

    with torch.no_grad():
        for y in ys:
            for x in xs:
                pin = xp[:, :, y:y+patch, x:x+patch]
                with torch.amp.autocast("cuda", enabled=CFG.USE_AMP):
                    cp = (pin - model(make_input(pin, level))).float()
                out[:, :, y:y+patch, x:x+patch] += cp * win
                wts[:, :, y:y+patch, x:x+patch] += win

    return (out / wts.clamp(min=1e-6))[:, :, :H, :W]

print("OK inference | conditioned sliding window")

## Test set (synthetic noise → metrics + DICOM)

In [ ]:
for ckpt in [BEST_CKPT, LAST_CKPT]:
    if os.path.exists(ckpt):
        load_ckpt(ckpt, model)
        print(f"Checkpoint | loaded={os.path.basename(ckpt)}"); break
else:
    print("No checkpoint — using the current weights.")

results = []
for i, path in enumerate(test_paths):
    arr, _ = load_dicom_normalized(path)
    mask   = load_or_compute_mask(path, arr)
    ct     = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).float().to(device)

    torch.manual_seed(9000 + i); np.random.seed(9000 + i)
    noisy_t, level = noise_adder(ct)

    den_t = sliding_window_inference(
        model, noisy_t, level, patch=CFG.PATCH_SIZE, overlap=CFG.PATCH_SIZE // 4
    ).clamp(0, 1)

    mt       = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)
    psnr_in  = masked_psnr(noisy_t, ct, mt).item()
    psnr_out = masked_psnr(den_t,   ct, mt).item()

    case_dir = os.path.join(CFG.OUT_DIR, f"test_{i+1:03d}")
    os.makedirs(case_dir, exist_ok=True)
    save_dicom_from_norm(arr, path, os.path.join(case_dir, "clean.dcm"))
    save_dicom_from_norm(noisy_t.squeeze().cpu().numpy(), path, os.path.join(case_dir, "noisy.dcm"))
    save_dicom_from_norm(den_t.squeeze().cpu().numpy(), path, os.path.join(case_dir, "denoised.dcm"))

    results.append({"path": path, "psnr_in": psnr_in, "psnr_out": psnr_out,
                    "gain": psnr_out - psnr_in, "level": float(level.mean())})
    print(f"test={i+1:03d}/{len(test_paths)} | psnr={psnr_in:.2f}->{psnr_out:.2f} | gain={psnr_out - psnr_in:+.2f}dB")

if results:
    g  = np.array([r["gain"]     for r in results])
    pi = np.array([r["psnr_in"]  for r in results])
    po = np.array([r["psnr_out"] for r in results])
    print(
        f"OK test | n={len(results)} | noisy={pi.mean():.2f}±{pi.std():.2f}dB | "
        f"denoised={po.mean():.2f}±{po.std():.2f}dB | gain={g.mean():+.2f}dB"
    )
    with open(os.path.join(CFG.LOG_DIR, "test_results.json"), "w") as f:
        json.dump(results, f, indent=1, default=float)

## Denoise a chosen DICOM (tune the strength)

Set `USER_DICOM_PATH` and `INFER_LEVEL` (local override). With
`ADD_SYNTHETIC_NOISE=True` synthetic noise is added (for testing), with `False`
the file is denoised as is (e.g. a real tomo slice). **`INFER_LEVEL` is the
knob**: higher = stronger denoising (risk of smoothing), lower = gentler
(keeps more texture).

In [ ]:
USER_DICOM_PATH     = test_paths[0] if test_paths else dicom_paths[0]
ADD_SYNTHETIC_NOISE = True
INFER_LEVEL         = CFG.INFER_LEVEL   # <-- SET the denoising strength HERE

for ckpt in [BEST_CKPT, LAST_CKPT]:
    if os.path.exists(ckpt):
        load_ckpt(ckpt, model); print(f"Checkpoint | {os.path.basename(ckpt)}"); break
model.eval()

print(f"DICOM | {os.path.basename(USER_DICOM_PATH)} | infer_level={INFER_LEVEL}")
arr, _  = load_dicom_normalized(USER_DICOM_PATH)
clean_t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).float().to(device)
input_t = clean_t

if ADD_SYNTHETIC_NOISE:
    torch.manual_seed(123); np.random.seed(123)
    input_t, _lv = noise_adder(clean_t)
    print(f"Noise | synthetic level={float(_lv.mean()):.4f}")

den_t = sliding_window_inference(
    model, input_t, INFER_LEVEL, patch=CFG.PATCH_SIZE, overlap=CFG.PATCH_SIZE // 4
).clamp(0, 1)

out_dir = os.path.join(CFG.OUT_DIR, "single")
os.makedirs(out_dir, exist_ok=True)
if ADD_SYNTHETIC_NOISE:
    save_dicom_from_norm(arr, USER_DICOM_PATH, os.path.join(out_dir, "clean.dcm"))
save_dicom_from_norm(input_t.squeeze().detach().cpu().numpy(), USER_DICOM_PATH, os.path.join(out_dir, "noisy.dcm"))
save_dicom_from_norm(den_t.squeeze().detach().cpu().numpy(),   USER_DICOM_PATH, os.path.join(out_dir, "denoised.dcm"))
print(f"OK | DICOMs saved in {out_dir}")

## Automatic trio (downloadable ZIP)

Takes the first file of the test set, adds synthetic noise, exports `clean/noisy/denoised.dcm`
to a ZIP and downloads it. Change `INFER_LEVEL_TRIO` to tune the denoising strength.

In [ ]:
INFER_LEVEL_TRIO = CFG.INFER_LEVEL   # <-- denoising strength

for ckpt in [BEST_CKPT, LAST_CKPT]:
    if os.path.exists(ckpt):
        load_ckpt(ckpt, model); print(f"Checkpoint | {os.path.basename(ckpt)}"); break
model.eval()

img_path = test_paths[0] if test_paths else dicom_paths[0]
print(f"DICOM | {os.path.basename(img_path)} | infer_level={INFER_LEVEL_TRIO}")

arr, _ = load_dicom_normalized(img_path)
clean_t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).float().to(device)
torch.manual_seed(123); np.random.seed(123)
noisy_t, _lv = noise_adder(clean_t)
print(f"Noise | synthetic level={float(_lv.mean()):.4f}")

den_t = sliding_window_inference(
    model, noisy_t, INFER_LEVEL_TRIO, patch=CFG.PATCH_SIZE, overlap=CFG.PATCH_SIZE // 4
).clamp(0, 1)

trio_dir = "/content/trio"
if os.path.isdir(trio_dir): shutil.rmtree(trio_dir)
os.makedirs(trio_dir, exist_ok=True)
save_dicom_from_norm(arr,                                      img_path, os.path.join(trio_dir, "clean.dcm"))
save_dicom_from_norm(noisy_t.squeeze().cpu().numpy(),          img_path, os.path.join(trio_dir, "noisy.dcm"))
save_dicom_from_norm(den_t.squeeze().detach().cpu().numpy(),   img_path, os.path.join(trio_dir, "denoised.dcm"))

zip_path = "/content/trio.zip"
if os.path.exists(zip_path): os.remove(zip_path)
shutil.make_archive("/content/trio", "zip", trio_dir)
print(f"OK trio | {zip_path}")
if IN_COLAB:
    from google.colab import files; files.download(zip_path)

## Trio from a chosen DICOM (downloadable ZIP)

Set `USER_DICOM_PATH` to the path of your file. With `ADD_SYNTHETIC_NOISE=True`
synthetic noise is added (useful for quantitative evaluation), with `False` the
file is denoised as is (e.g. a real tomo slice or the phantom).

In [ ]:
USER_DICOM_PATH     = "/content/projection.dcm"  # <-- CHANGE HERE
ADD_SYNTHETIC_NOISE = False   # True = add synthetic noise; False = denoise as is
INFER_LEVEL_USER    = CFG.INFER_LEVEL   # <-- denoising strength

if not os.path.exists(USER_DICOM_PATH):
    raise FileNotFoundError(f"DICOM not found: {USER_DICOM_PATH}")

for ckpt in [BEST_CKPT, LAST_CKPT]:
    if os.path.exists(ckpt):
        load_ckpt(ckpt, model); print(f"Checkpoint | {os.path.basename(ckpt)}"); break
model.eval()

print(f"DICOM | {os.path.basename(USER_DICOM_PATH)} | infer_level={INFER_LEVEL_USER}")
arr, _ = load_dicom_normalized(USER_DICOM_PATH)
clean_t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).float().to(device)
input_t = clean_t

if ADD_SYNTHETIC_NOISE:
    torch.manual_seed(123); np.random.seed(123)
    input_t, _lv = noise_adder(clean_t)
    print(f"Noise | synthetic level={float(_lv.mean()):.4f}")

den_t = sliding_window_inference(
    model, input_t, INFER_LEVEL_USER, patch=CFG.PATCH_SIZE, overlap=CFG.PATCH_SIZE // 4
).clamp(0, 1)

trio_dir = "/content/trio_user"
if os.path.isdir(trio_dir): shutil.rmtree(trio_dir)
os.makedirs(trio_dir, exist_ok=True)
if ADD_SYNTHETIC_NOISE:
    save_dicom_from_norm(arr, USER_DICOM_PATH, os.path.join(trio_dir, "clean.dcm"))
save_dicom_from_norm(input_t.squeeze().detach().cpu().numpy(), USER_DICOM_PATH, os.path.join(trio_dir, "noisy.dcm"))
save_dicom_from_norm(den_t.squeeze().detach().cpu().numpy(),   USER_DICOM_PATH, os.path.join(trio_dir, "denoised.dcm"))

zip_path = "/content/trio_user.zip"
if os.path.exists(zip_path): os.remove(zip_path)
shutil.make_archive("/content/trio_user", "zip", trio_dir)
print(f"OK trio | {zip_path}")
if IN_COLAB:
    from google.colab import files; files.download(zip_path)

## Denoise an already noisy DICOM (downloadable ZIP)

Already noisy DICOM as input (e.g. a real tomo projection): it is denoised directly,
without adding synthetic noise. Exports `noisy.dcm` and `denoised.dcm`.

In [ ]:
USER_DICOM_PATH  = "/content/projection.dcm"  # <-- CHANGE HERE
INFER_LEVEL_RAW  = CFG.INFER_LEVEL   # <-- denoising strength (lower = gentler)

if not os.path.exists(USER_DICOM_PATH):
    raise FileNotFoundError(f"DICOM not found: {USER_DICOM_PATH}")

for ckpt in [BEST_CKPT, LAST_CKPT]:
    if os.path.exists(ckpt):
        load_ckpt(ckpt, model); print(f"Checkpoint | {os.path.basename(ckpt)}"); break
model.eval()

print(f"DICOM | {os.path.basename(USER_DICOM_PATH)} | infer_level={INFER_LEVEL_RAW}")
arr, _ = load_dicom_normalized(USER_DICOM_PATH)
input_t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).float().to(device)

den_t = sliding_window_inference(
    model, input_t, INFER_LEVEL_RAW, patch=CFG.PATCH_SIZE, overlap=CFG.PATCH_SIZE // 4
).clamp(0, 1)

out_dir = "/content/denoise_raw"
if os.path.isdir(out_dir): shutil.rmtree(out_dir)
os.makedirs(out_dir, exist_ok=True)
save_dicom_from_norm(input_t.squeeze().detach().cpu().numpy(), USER_DICOM_PATH, os.path.join(out_dir, "noisy.dcm"))
save_dicom_from_norm(den_t.squeeze().detach().cpu().numpy(),   USER_DICOM_PATH, os.path.join(out_dir, "denoised.dcm"))

zip_path = "/content/denoise_raw.zip"
if os.path.exists(zip_path): os.remove(zip_path)
shutil.make_archive("/content/denoise_raw", "zip", out_dir)
print(f"OK denoise | {zip_path}")
if IN_COLAB:
    from google.colab import files; files.download(zip_path)

## INFER_LEVEL sweep on the phantom (choosing the best level)

Loads an already noisy tomo/phantom DICOM, tries N values of `INFER_LEVEL`,
shows a grid of previews + an `hf` / `std_patch` table, and downloads a ZIP with
all the DICOMs to open them in a viewer. Use it to choose the level without
retraining.

In [ ]:
PHANTOM_PATH  = "/content/Tomo_6.dcm"   # <-- CHANGE to your file
LEVELS_TO_TEST  = [0.001, 0.002, 0.004, 0.006, 0.008, 0.010, 0.015, 0.020]
PREVIEW_SIZE    = 512   # preview crop size (pixels)

import matplotlib.pyplot as plt

if not os.path.exists(PHANTOM_PATH):
    raise FileNotFoundError(f"File not found: {PHANTOM_PATH}")

for ckpt in [BEST_CKPT, LAST_CKPT]:
    if os.path.exists(ckpt):
        load_ckpt(ckpt, model); print(f"Checkpoint | {os.path.basename(ckpt)}"); break
model.eval()

arr, _  = load_dicom_normalized(PHANTOM_PATH)
mask    = compute_breast_mask(arr)
input_t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).float().to(device)
mask_t  = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)

ys, xs = np.where(mask)
cy, cx = (int(ys.mean()), int(xs.mean())) if len(ys) else (arr.shape[0]//2, arr.shape[1]//2)
p  = PREVIEW_SIZE
y0 = int(np.clip(cy - p//2, 0, arr.shape[0] - p))
x0 = int(np.clip(cx - p//2, 0, arr.shape[1] - p))

def _crop(img): return img[y0:y0+p, x0:x0+p]

results_sweep, denoised_tensors = [], []
print(f"Sweep | {os.path.basename(PHANTOM_PATH)} | levels={LEVELS_TO_TEST}")

for lv in LEVELS_TO_TEST:
    den_t = sliding_window_inference(
        model, input_t, lv, patch=CFG.PATCH_SIZE, overlap=CFG.PATCH_SIZE // 4
    ).clamp(0, 1)
    denoised_tensors.append(den_t)
    hf        = masked_hf_ratio(den_t, input_t, mask_t).item()
    std_patch = float(_crop(den_t.squeeze().cpu().numpy()).std())
    results_sweep.append({"level": lv, "hf": hf, "std_patch": std_patch})
    print(f"  level={lv:.3f} | hf={hf:.3f} | std_patch={std_patch:.4f}")

noisy_std = float(_crop(arr).std())
print(f"\n{'level':>8}  {'hf':>6}  {'std_patch':>10}  {'std%noisy':>10}")
print("-" * 42)
for r in results_sweep:
    print(f"{r['level']:>8.3f}  {r['hf']:>6.3f}  {r['std_patch']:>10.4f}  {r['std_patch']/noisy_std*100:>9.1f}%")

noisy_crop = _crop(arr)
vmin, vmax = noisy_crop.min(), noisy_crop.max()
n_tot  = len(LEVELS_TO_TEST) + 1
n_cols = 3
n_rows = math.ceil(n_tot / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
axes = axes.flatten()

axes[0].imshow(noisy_crop, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("noisy (input)", fontsize=10)
axes[0].axis("off")

for i, (lv, den_t) in enumerate(zip(LEVELS_TO_TEST, denoised_tensors)):
    img = _crop(den_t.squeeze().cpu().numpy())
    axes[i+1].imshow(img, cmap="gray", vmin=vmin, vmax=vmax)
    axes[i+1].set_title(f"level={lv:.3f}  hf={results_sweep[i]['hf']:.3f}", fontsize=10)
    axes[i+1].axis("off")

for j in range(n_tot, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Sweep INFER_LEVEL — {os.path.basename(PHANTOM_PATH)}", fontsize=12)
plt.tight_layout(); plt.show()

sweep_dir = "/content/sweep_level"
if os.path.isdir(sweep_dir): shutil.rmtree(sweep_dir)
os.makedirs(sweep_dir, exist_ok=True)
save_dicom_from_norm(arr, PHANTOM_PATH, os.path.join(sweep_dir, "noisy.dcm"))
for lv, den_t in zip(LEVELS_TO_TEST, denoised_tensors):
    fname = f"denoised_lv{int(round(lv*1000)):04d}.dcm"
    save_dicom_from_norm(den_t.squeeze().cpu().numpy(), PHANTOM_PATH,
                         os.path.join(sweep_dir, fname))

zip_path = "/content/sweep_level.zip"
if os.path.exists(zip_path): os.remove(zip_path)
shutil.make_archive("/content/sweep_level", "zip", sweep_dir)
print(f"\nOK sweep | ZIP={zip_path}")
if IN_COLAB:
    from google.colab import files; files.download(zip_path)

## Denoise 11 tomo projections (output for reconstruction)

Denoises the 11 projections and saves the DICOMs with their **original names unchanged**
(`Tomo_1.dcm` … `Tomo_11.dcm`) in a `tomo_denoised/` folder.
The downloaded ZIP contains only the denoised files, ready for reconstruction.

In [ ]:
# --- CONFIGURATION ---
# Upload Tomo_1.dcm ... Tomo_11.dcm to /content/ before running.
# The names in TOMO_FILES must match the uploaded files EXACTLY.

TOMO_DIR    = "/content"   # folder with the uploaded DICOMs
INFER_LEVEL = CFG.INFER_LEVEL   # denoising strength (0.003, reference value)

TOMO_FILES = [
    "Tomo_1.dcm",
    "Tomo_2.dcm",
    "Tomo_3.dcm",
    "Tomo_4.dcm",
    "Tomo_5.dcm",
    "Tomo_6.dcm",
    "Tomo_7.dcm",
    "Tomo_8.dcm",
    "Tomo_9.dcm",
    "Tomo_10.dcm",
    "Tomo_11.dcm",
]

# ----------------------------------------------------------------------
for ckpt in [BEST_CKPT, LAST_CKPT]:
    if os.path.exists(ckpt):
        load_ckpt(ckpt, model); print(f"Checkpoint | {os.path.basename(ckpt)}"); break
model.eval()

out_dir = "/content/tomo_denoised"
if os.path.isdir(out_dir): shutil.rmtree(out_dir)
os.makedirs(out_dir)

# preliminary check: all files exist before starting
missing = [f for f in TOMO_FILES if not os.path.exists(os.path.join(TOMO_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing files in {TOMO_DIR}: {missing}")

print(f"Denoise {len(TOMO_FILES)} projections | level={INFER_LEVEL} | out={out_dir}")
print("-" * 60)

for fname in TOMO_FILES:
    src = os.path.join(TOMO_DIR, fname)
    dst = os.path.join(out_dir, fname)   # same name, different folder

    arr, _ = load_dicom_normalized(src)
    t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).float().to(device)

    den_t = sliding_window_inference(
        model, t, INFER_LEVEL, patch=CFG.PATCH_SIZE, overlap=CFG.PATCH_SIZE // 4
    ).clamp(0, 1)

    save_dicom_from_norm(den_t.squeeze().cpu().numpy(), src, dst)
    print(f"  OK  {fname}  ->  tomo_denoised/{fname}")

# final check: all 11 files were written
written = sorted(os.listdir(out_dir))
assert written == sorted(TOMO_FILES), f"ERROR in output names: {written}"

print("-" * 60)
print(f"OK | {len(written)} file in tomo_denoised/: {written}")

zip_path = "/content/tomo_denoised.zip"
if os.path.exists(zip_path): os.remove(zip_path)
shutil.make_archive("/content/tomo_denoised", "zip", out_dir)
print(f"ZIP | {zip_path}")
if IN_COLAB:
    from google.colab import files; files.download(zip_path)